# PoB AI Planner — avaliação holdout e promoção manual

Esta etapa lê o holdout, calcula ranking Spearman, recall Pareto, redução de recálculos e latência. O benchmark nativo de 1.000 propostas precisa ser medido no mesmo checkout/patch do PoB e informado abaixo. Um relatório que falha em qualquer gate permanece em staging.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, sys, subprocess, json
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/00 - Projetos/PoB - IA')
SOURCE = Path('/content/pob_ai_source'); SOURCE.mkdir(exist_ok=True)
with zipfile.ZipFile(ROOT / '05_Notebooks/pob_gnn_source.zip') as archive: archive.extractall(SOURCE)
sys.path.insert(0, str(SOURCE))
DATA = ROOT / '02_Validated_Datasets'
STAGING = ROOT / '03_Models_Staging'
PACKAGE_ID = 'COLE_AQUI_O_ID_staging_'
PACKAGE = STAGING / PACKAGE_ID
native_exact_1000_seconds = float(input('Benchmark local medido: segundos para 1.000 recalculos nativos no mesmo patch: '))
ipc_overhead_seconds = float(input('IPC + serialização medida para 1.000 propostas (segundos): '))
result = subprocess.run([sys.executable, '-m', 'tools.optimizer.gnn.evaluate', '--package', str(PACKAGE), '--data-root', str(DATA), '--native-exact-1000-seconds', str(native_exact_1000_seconds), '--ipc-overhead-seconds', str(ipc_overhead_seconds)], cwd=SOURCE, capture_output=True, text=True)
print(result.stdout)
if result.returncode not in (0, 2): raise RuntimeError(result.stderr)
report = json.loads((PACKAGE / 'evaluation_report.json').read_text())
print({name: gate['passed'] for name, gate in report['gates'].items()})

## Promoção exige ação humana
Confira os cinco valores, a skill, cenário, fingerprint e proveniência. A variável abaixo vem desativada. Só ative depois de confirmar que este pacote e estas autorizações pertencem ao contexto esperado.

In [ ]:
PROMOTE_TO_APPROVED = False
if PROMOTE_TO_APPROVED:
    if report.get('status') != 'passed': raise RuntimeError('Gate falhou; pacote permanece em staging')
    subprocess.run([sys.executable, '-m', 'tools.optimizer.gnn.promote', '--staging-package', str(PACKAGE), '--approved-root', str(ROOT / '04_Models_Approved')], cwd=SOURCE, check=True)
else:
    print('Sem promoção: altere PROMOTE_TO_APPROVED somente após revisão manual de um relatório aprovado.')